In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load the breast cancer dataset and create train, validation, and test splits
data = load_breast_cancer()
X, y = data.data, data.target

# Split into 70% train, 15% validation, 15% test (stratified to preserve class balance)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

# 2. Train Logistic Regression as the baseline and record its accuracy[cite: 1]
baseline_model = LogisticRegression(max_iter=5000, random_state=42)
baseline_model.fit(X_train, y_train)

val_pred = baseline_model.predict(X_val)
baseline_val_acc = accuracy_score(y_val, val_pred)
baseline_test_acc = accuracy_score(y_test, baseline_model.predict(X_test))

print(f"Baseline Validation Accuracy: {baseline_val_acc:.4f}")
print(f"Baseline Test Accuracy:       {baseline_test_acc:.4f}")

# 3. Use 5-fold cross-validation to estimate performance[cite: 1]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(baseline_model, X_train, y_train, cv=cv, scoring="accuracy")
cv_mean_acc = cv_scores.mean()

print(f"5-Fold CV Mean Accuracy:      {cv_mean_acc:.4f} (+/- {cv_scores.std():.4f})")

# 4. Create a pipeline with StandardScaler and Logistic Regression[cite: 1]
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000, random_state=42)),
])

# 5. Use GridSearchCV to tune at least one hyperparameter and compare results[cite: 1]
param_grid = {
    "clf__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "clf__solver": ["lbfgs", "liblinear"],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
tuned_val_acc = accuracy_score(y_val, best_model.predict(X_val))
tuned_test_acc = accuracy_score(y_test, best_model.predict(X_test))

print(f"\nBest Parameters:              {grid_search.best_params_}")
print(f"Tuned CV Mean Accuracy:       {grid_search.best_score_:.4f}")
print(f"Tuned Validation Accuracy:    {tuned_val_acc:.4f}")
print(f"Tuned Test Accuracy:          {tuned_test_acc:.4f}")

# Generate submission summary table[cite: 1]
summary_df = pd.DataFrame({
    "Model Stage": [
        "1. Baseline Logistic Regression",
        "2. 5-Fold Cross-Validation (Unscaled)",
        "3. Tuned Pipeline (StandardScaler + GridSearchCV)",
    ],
    "CV / Validation Score": [
        round(baseline_val_acc, 4),
        round(cv_mean_acc, 4),
        round(grid_search.best_score_, 4),
    ],
    "Test Accuracy": [
        round(baseline_test_acc, 4),
        "—",
        round(tuned_test_acc, 4),
    ],
})

print("\n--- Submission Comparison Table ---")
print(summary_df.to_string(index=False))

Baseline Validation Accuracy: 0.9765
Baseline Test Accuracy:       0.9186
5-Fold CV Mean Accuracy:      0.9524 (+/- 0.0254)

Best Parameters:              {'clf__C': 0.1, 'clf__solver': 'lbfgs'}
Tuned CV Mean Accuracy:       0.9849
Tuned Validation Accuracy:    0.9882
Tuned Test Accuracy:          0.9767

--- Submission Comparison Table ---
                                      Model Stage  CV / Validation Score Test Accuracy
                  1. Baseline Logistic Regression                 0.9765        0.9186
            2. 5-Fold Cross-Validation (Unscaled)                 0.9524             —
3. Tuned Pipeline (StandardScaler + GridSearchCV)                 0.9849        0.9767
